# Tipsomaly zero-shot natural-corruption benchmark

This notebook runs the official Tipsomaly inference path through the shared MVTec AD / VisA corruption harness. It uses the checkpoint trained on the other dataset, TIPS-L/14-HR, decoupled fixed/learned prompts, spatial-token plus local-evidence image scoring, and the official Gaussian-smoothed pixel map. Enable a Kaggle GPU and Internet, or attach the three TIPS base-model files listed below as a Kaggle input.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

print("====== STEP 1: CLONING BENCHMARK AND OFFICIAL TIPSOMALY ======")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
TIPSOMALY_ROOT = Path("/kaggle/working/Tipsomaly")

def clone_or_update(url, destination):
    if not destination.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", url, str(destination)],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(destination), "pull", "--ff-only"],
            check=True,
        )

clone_or_update(
    f"https://github.com/{BENCHMARK_REPOSITORY}.git", BENCHMARK_ROOT
)
clone_or_update(
    "https://github.com/Alireza99Salehi/Tipsomaly.git", TIPSOMALY_ROOT
)

print("\n====== STEP 2: INSTALLING MINIMAL TIPSOMALY DEPENDENCIES ======")
# Preserve Kaggle's CUDA-enabled torch/torchvision instead of installing the
# upstream frozen environment. JAX is imported by upstream helper modules but
# is not used by this TIPS-backbone inference path.
packages = ["sentencepiece==0.2.1", "scipy>=1.9"]
if importlib.util.find_spec("jax") is None:
    packages.append("jax[cpu]>=0.4,<0.7")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

required_files = [
    TIPSOMALY_ROOT / "model" / "tips" / "load_model.py",
    TIPSOMALY_ROOT / "model" / "omaly" / "text_encoder.py",
    TIPSOMALY_ROOT / "workspaces" / "trained_on_mvtec_default",
    TIPSOMALY_ROOT / "workspaces" / "trained_on_visa_default",
]
missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f"Incomplete Tipsomaly clone; missing: {missing}")
os.environ["TIPSOMALY_ROOT"] = str(TIPSOMALY_ROOT)
print(f"Benchmark repository: {BENCHMARK_ROOT}")
print(f"Official Tipsomaly:   {TIPSOMALY_ROOT}")
print("Environment ready.")

The default is the categorized corruption protocol. Set `USE_CATEGORIZED_CORRUPTIONS = False` for every concrete corruption independently. For an offline run, attach `tokenizer.model`, `tips_oss_l14_highres_distilled_vision.npz`, and `tips_oss_l14_highres_distilled_text.npz`; the next cell discovers them automatically.

In [ ]:
import gc
import os
import shutil
import sys
from pathlib import Path

import torch

BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
HARNESS_ROOT = BENCHMARK_ROOT / "zero_shot"
TIPSOMALY_ROOT = Path("/kaggle/working/Tipsomaly")
for import_path in (BENCHMARK_ROOT, HARNESS_ROOT, TIPSOMALY_ROOT):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ["TIPSOMALY_ROOT"] = str(TIPSOMALY_ROOT)

from shared import corruption_plan_path
from harness.config import CLEAN_CONDITION
from harness.runner import run_evaluation

# Choose exactly one evaluation target.
# DATASET_NAME = "mvtec"
DATASET_NAME = "visa"
MODEL_NAME = "Tipsomaly"
MODEL_VERSION = "l14h"
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa"}:
    raise ValueError("DATASET_NAME must be either 'mvtec' or 'visa'.")
IS_MVTEC = DATASET_NAME == "mvtec"
SELECTED_DATASET = "MVTec AD" if IS_MVTEC else "VisA"
# Official zero-shot protocol: learn prompts on the other benchmark.
WEIGHT_DATASET = "visa" if IS_MVTEC else "mvtec"
if WEIGHT_DATASET == DATASET_NAME:
    raise RuntimeError("Tipsomaly checkpoint leakage detected.")

USE_CATEGORIZED_CORRUPTIONS = True
CATEGORIZED_CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "motion_blur", "zoom_blur",
    "brightness", "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = ["noise", "blur", "photometric", "geometric"]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS
    else UNCATEGORIZED_CORRUPTION_TYPES
)
INCLUDE_CLEAN_BASELINE = True
ZERO_CORRUPTION_CONDITION = CLEAN_CONDITION
SEVERITY_LEVELS = [1, 2, 3, 4]
BATCH_SIZE = 2  # Safe default for TIPS-L/14-HR on a 16 GB Kaggle GPU.
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"

MVTEC_PATH = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_PATH = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Enable a Kaggle GPU accelerator for Tipsomaly.")

CORRUPTION_PLAN = corruption_plan_path(DATASET_NAME)
if USE_CATEGORIZED_CORRUPTIONS and not CORRUPTION_PLAN.exists():
    raise FileNotFoundError(f"Categorized corruption plan not found: {CORRUPTION_PLAN}")

# The small learned prompt checkpoints are committed to the official repo.
TIPSOMALY_CHECKPOINT = (
    TIPSOMALY_ROOT
    / "workspaces"
    / f"trained_on_{WEIGHT_DATASET}_default"
    / "vegan-arkansas"
    / "checkpoints"
    / "learnable_params_2.pth"
)
if not TIPSOMALY_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Released cross-dataset checkpoint not found: {TIPSOMALY_CHECKPOINT}"
    )

# Reuse attached base components when present; otherwise the official TIPS
# loader downloads these exact files from storage.googleapis.com/tips_data.
TIPS_MODELS_DIR = Path("/kaggle/working/tips")
TIPS_MODELS_DIR.mkdir(parents=True, exist_ok=True)
TIPS_BASE_FILES = [
    "tokenizer.model",
    "tips_oss_l14_highres_distilled_vision.npz",
    "tips_oss_l14_highres_distilled_text.npz",
]
input_root = Path("/kaggle/input")
for filename in TIPS_BASE_FILES:
    destination = TIPS_MODELS_DIR / filename
    if destination.exists():
        continue
    attached = next(input_root.rglob(filename), None) if input_root.exists() else None
    if attached is not None:
        try:
            destination.symlink_to(attached)
        except OSError:
            shutil.copy2(attached, destination)
        print(f"Using attached TIPS component: {attached}")

model_kwargs = {
    MODEL_NAME: {
        "tipsomaly_root": str(TIPSOMALY_ROOT),
        "checkpoint_path": str(TIPSOMALY_CHECKPOINT),
        "models_dir": str(TIPS_MODELS_DIR),
        "dataset_name": DATASET_NAME,
        "weight_dataset": WEIGHT_DATASET,
        "model_version": MODEL_VERSION,
        "image_size": 518,
        "sigma": 4,
        "epoch": 2,
        "fixed_prompt_type": "industrial",
        "prompt_learn_method": "concat",
        "n_prompt": 8,
        "decoupled_prompt": True,
        "aggregate_local2global": True,
    }
}

print("LAUNCHING TIPSOMALY ROBUSTNESS BENCHMARK")
print(f"Evaluation target:  {SELECTED_DATASET}")
print(f"Weights trained on: {WEIGHT_DATASET} (cross-dataset zero-shot)")
print(f"Prompt checkpoint:  {TIPSOMALY_CHECKPOINT}")
print(f"TIPS base:          {TIPS_MODELS_DIR} ({MODEL_VERSION})")
print(f"Zero corruption:    {ZERO_CORRUPTION_CONDITION if INCLUDE_CLEAN_BASELINE else 'disabled'}")
print(f"Corruptions:        {CORRUPTION_TYPES} @ {SEVERITY_LEVELS}")
print(f"Categorized:        {USE_CATEGORIZED_CORRUPTIONS}")
print(f"Plan:               {CORRUPTION_PLAN}")
print(f"Device/batch:       {DEVICE} / {BATCH_SIZE}")
print(f"Outputs:            {OUTPUT_ROOT}")

run_evaluation(
    mvtec_root=MVTEC_PATH if IS_MVTEC else None,
    visa_root=None if IS_MVTEC else VISA_PATH,
    output_root=OUTPUT_ROOT,
    models=[MODEL_NAME],
    model_kwargs=model_kwargs,
    device=DEVICE,
    dataset=DATASET_NAME,
    corruption_types=CORRUPTION_TYPES,
    severity_levels=SEVERITY_LEVELS,
    include_clean=INCLUDE_CLEAN_BASELINE,
    batch_size=BATCH_SIZE,
    corruption_cache_root=CORRUPTION_CACHE_ROOT,
    corruption_cache_format=CORRUPTION_CACHE_FORMAT,
    categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
    categorized_corruption_plans={DATASET_NAME: str(CORRUPTION_PLAN)},
    corruption_seed=(
        CATEGORIZED_CORRUPTION_SEED if USE_CATEGORIZED_CORRUPTIONS else None
    ),
)

gc.collect()
torch.cuda.empty_cache()
print(f"Tipsomaly evaluation complete. Outputs: {OUTPUT_ROOT}")